In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg, to_date

spark = SparkSession.builder.appName("RollingAvgBalance").getOrCreate()

# Sample Data
data = [
    ("A1", "2024-01-01", 1000),
    ("A1", "2024-01-02", 1200),
    ("A1", "2024-01-03", 1100),
    ("A1", "2024-01-04", 1500),
    ("A1", "2024-01-05", 1300),
    ("A1", "2024-01-06", 1400),
    ("A1", "2024-01-07", 1600),
    ("A1", "2024-01-08", 1700)
]

columns = ["account_id", "date", "balance"]

df = spark.createDataFrame(data, columns) \
          .withColumn("date", to_date("date"))

# Window: last 7 days
window_spec = Window.partitionBy("account_id") \
                    .orderBy("date") \
                    .rowsBetween(-6, 0)

# Rolling average
result_df = df.withColumn(
    "rolling_avg_balance",
    avg("balance").over(window_spec)
)

result_df.show()
